# Automatically find all person names in your reference lists and paragraphs and save them with their source file and volume information.

I have decided to work with spacy´s en_core_web_trf model.

For one, this model uses a real language model underneath (RoBERTa), additionally, while sm and lg use statistical features and word vectors, trf runs a full transformer that reads the whole sentence at once so it understands context.

Furthermore, en_core_web_trf has the highest NER accuracy of all spaCy English models and it is better on rare and theory-world names such as Lacan, Irigaray, and Kristeva, which rarely appear in general training data. The transformer generalises better to uncommon names by reading surrounding context.

Additionally, because this corpus has fragmented sentences (e.g. "ness fading as soon as the shared task was done."), the choice was to use en_core_web_trf because Transformers are more robust to unusual token sequences than the statistical models. 

However, one trade-off one needs to be aware of is that this model is significantly slower, however, in this case, this will only take several minutes and not hours on end. 




For this Notebook, I have used the following documentation to help me write the code: 
- [Named Entity Recognition](https://spacy.io/usage/linguistic-features#named-entities)
- [English Model Comparison](https://spacy.io/models/en)
- [Counter for Counting Name Frequencies](https://docs.python.org/3/library/collections.html#collections.Counter)
- [Pandas for downloading and reshaping data](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)
- [tqdm for the progress bar](https://tqdm.github.io/)

#### 1. Imports

I decided for spacy [because](https://www.geeksforgeeks.org/python/python-named-entity-recognition-ner-using-spacy/): 

In [ ]:
import pandas as pd
import spacy
import spacy_curated_transformers  # I need to load en_core_web_trf
from collections import Counter
from tqdm import tqdm

# this lets pandas show a progress bar when I use .apply()
tqdm.pandas()

# load the transformer model — more accurate than the small model
# run once in terminal if not installed:
# pip install spacy-transformers
# python -m spacy download en_core_web_trf
nlp = spacy.load("en_core_web_trf")

#### 2. Loading the H2 Corpus

In [4]:
df = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/07_h2_normalized.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nRegion types:")
print(df["region_type"].value_counts())

Shape: (19703, 5)

Columns: ['source_file', 'page_number', 'region_type', 'text', 'tokens']

Region types:
region_type
paragraph         19286
reference_list      417
Name: count, dtype: int64


#### 3. Adding Issue Number and Volume Label

In [5]:
import re

def get_issue_number(filepath):
    """Extract the issue number from the filename, e.g. heresies_08_combined → 8"""
    match = re.search(r"heresies_(\d+)_combined", filepath)
    return int(match.group(1)) if match else None

def get_volume(issue_nr):
    """Map issue number to a volume label"""
    if issue_nr is None:
        return None
    if issue_nr <= 4:  return "Vol1_1977-1978"
    if issue_nr <= 8:  return "Vol2_1978-1979"
    if issue_nr <= 12: return "Vol3_1980-1981"
    if issue_nr <= 16: return "Vol4_1981-1983"
    if issue_nr <= 20: return "Vol5_1984-1985"
    if issue_nr <= 24: return "Vol6_1987-1989"
    if issue_nr <= 27: return "Vol7_1990-1993"
    return None

df["issue"]  = df["source_file"].apply(get_issue_number)
df["volume"] = df["issue"].apply(get_volume)

print(df["volume"].value_counts())

volume
Vol2_1978-1979    4284
Vol1_1977-1978    3543
Vol3_1980-1981    3467
Vol4_1981-1983    2748
Vol6_1987-1989    2275
Vol5_1984-1985    2231
Vol7_1990-1993    1155
Name: count, dtype: int64


#### 4. Extracting person names with Spacy (paragraphs and reference_list separately)

In [16]:
def extract_persons(text):
    if pd.isna(text):
        return []
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

# split into two dataframes
df_para = df[df["region_type"] == "paragraph"].copy()
df_refs = df[df["region_type"] == "reference_list"].copy()

print(f"Paragraphs: {len(df_para)} rows")
print(f"Reference lists: {len(df_refs)} rows")

# run NER on each separately
df_para["persons"] = df_para["text"].progress_apply(extract_persons)
df_refs["persons"] = df_refs["text"].progress_apply(extract_persons)

Paragraphs: 19286 rows
Reference lists: 417 rows


100%|██████████| 417/417 [00:09<00:00, 41.83it/s]


#### 5. Basic Cleaning and Removing Noise (different rules for paragraph and reference_list)

In [22]:
def clean_para(persons):
    cleaned = []
    for name in persons:
        name = name.replace("'s", "").strip()
        if any(char.isdigit() for char in name):
            continue
        # paragraphs: require 2+ words (hyphens count)
        if len(name.replace("-", " ").split()) < 2:
            continue
        cleaned.append(name)
    return cleaned

def clean_refs(persons):
    cleaned = []
    for name in persons:
        name = name.replace("'s", "").strip()
        if any(char.isdigit() for char in name):
            continue
        # same 2-word rule as paragraphs — hyphens count as word separators
        if len(name.replace("-", " ").split()) < 2:
            continue
        cleaned.append(name)
    return cleaned

df_para["persons"] = df_para["persons"].apply(clean_para)
df_refs["persons"] = df_refs["persons"].apply(clean_refs)

#### 6. Checking results for both

In [67]:
para_persons = [name for persons in df_para["persons"] for name in persons]
refs_persons  = [name for persons in df_refs["persons"]  for name in persons]

print("=== PARAGRAPHS ===")
print(f"Total mentions: {len(para_persons)}, Unique: {len(set(para_persons))}")
for name, count in Counter(para_persons).most_common(20):
    print(f"  {name}: {count}")

print("\n=== REFERENCE LISTS ===")
print(f"Total mentions: {len(refs_persons)}, Unique: {len(set(refs_persons))}")
for name, count in Counter(refs_persons).most_common(20):
    print(f"  {name}: {count}")

=== PARAGRAPHS ===
Total mentions: 1134, Unique: 942
  Anais Nin: 8
  Bessie Smith: 7
  Ruth Crawford Seeger: 7
  Myrna Zimmerman: 6
  Melvonna Ballenger: 6
  Edie Lynch: 5
  Lucy Lippard: 5
  Virginia Woolf: 5
  Howardena Pindell: 4
  Alile Sharon Larkin: 4
  George Sand: 4
  Amédée Ozenfant: 4
  Adolf Loos: 4
  Natalie Barney: 4
  Collette Pean: 4
  Billie Holiday: 4
  Suzanne Lacy: 4
  Ida B. Wells-Barnett: 3
  Norma Jean Serena: 3
  James Van Der Zee: 3

=== REFERENCE LISTS ===
Total mentions: 172, Unique: 165
  Mary Oldham Eagle Kavanaugh: 2
  Mimi Smith: 2
  Adrienne Rich: 2
  Charlotte Bunch: 2
  Andrea Dworkin: 2
  Raymond Williams: 2
  Inez Garcia: 2
  Neomi Lorenzano: 1
  Margot Astrov: 1
  Eileen Southern: 1
  Mignon Holland: 1
  Carolyn Thomas: 1
  Judith Van Allen: 1
  Pat Lasch: 1
  Carter Ratcliff: 1
  Barbara Zucker: 1
  Diana Ross: 1
  Herbert Schiller: 1
  Ruby Rich: 1
  Lois Weber: 1


#### Cell 7 — Fix false positives and then combine both

In [66]:
# I filled the false positives by looking at the most common names in both lists and checking which ones are not actually people. 
# I also added some name variations to the mapping.
false_positives = {
    "Editorial Collective", "De Colores", "Sacred Song Sung",
    "Kitchen Dreams", "Kitchen Artists", "Kitchens", "Iris Films",
    "The BLACK WOMAN", "The Great Goddess",
    "F. Adler", "The Problematics of the Modern Chicana", "Little Souls in Pain", "the Chilena Melillan", 
    "The Black Scholar", "La Malinche", "The accomplishments of the",
    "Black Women", "Of the Black", "I am", "The people of the", 
    "Big Sweet", "The Queen of the Water", "ing Star", "The Happiness Mystique", 
    "Alan o", "i gla", "the Wizard of", "The women of the Commonwealth", 
    "The House of Meanings", "The Art of Cooking", "L. G.", "J. F.", "Spider Woman",
    "the Prince of Wales", "BETOERUNG DER BLAUEN MATROSEN", "Sweet Honey in the Rock", 
    "- Id rather", "The Pioneers", "The Socialist Women", "lust wh.", 
    "dat darknetch", "der queed", "ter wad", "woan vouch", "straitch widder", 
    "foamig i", "narror i", "iffer persob", "der queed", "mersouris eyelatches", 
    "ebry thins", "ther queed ob", "ebry tibe", "the Holy One", "the power of her ang", 
    "vir 'green gin", "Fred S.", "The BLACK WOM-", "Voice of Neighbor", "Voice of Reporter", 
    "Voice of Another Neighbor", "The Motif of Presentation", "quees in´s", "THE MOTHER", "THE DAUGHTER", 
    "Fu Man-", "ELECI KA", "eslie Artis", "The bread of the Mother House", "El Al", 
    "Tea Cake", "an Asian-American", "Butch fem", "The narcissist", "I Am Mirror", "Song of Hope", 
    "PEANUT BUTTER", "JELLY SANDWICH", "ing fe", "E. J.", "WOMEN IN THE SHAD-", "Black Afric", "The Bad Mother", 
    "beyond the gair", "The Twenties", "Violette de", "Bo-Peep", "THE MYTH", "vision H.D", "Mary S", "THE TREE OF LIFE", 
    "Leading Lady", "Mi Nombre", "MI NOMBRE", "the Hono", "Anonymous witness", "ed laterall", "My head doesn't", "The woman in love",
    "TECHQUA IKACHI", "Israel Ish-Rachel", "Writer of Songs", "the Old Ship of Zion", "the farr", "The Visionary", "The Artist", "The Lover", 
    "The Mother", "The Lesbian Body", "couid go", "hie trius", "a Christman Poeam", "Wonder Woman", "takene gowr wus", "The Fear of Taking", "The Fear of", 
    "erotica jus", "the dange", "noonday sun", "The Female Man", "mensch Womon", "The Seven of Us", "ing house of", "A perceptive", "THE SHOEMAKER", 
    "capable of the", "Les Guerilleres"  
}

name_mapping = {
    "Ida b. Wells":    "Ida B. Wells-Barnett",
    "Ida B. Wells":    "Ida B. Wells-Barnett",
    "Lucy R. Lippard": "Lucy Lippard",
    "Sharon Larkin":   "Alile Sharon Larkin",
    "B. Ruby Rich":    "Ruby Rich",
    "Adrienne Rich´s most": "Adrienne Rich",
    "Honor Moore´s Mourning Pictures": "Honor Moore",
    "Raymond Wil-": "Raymond Williams",
    "Christopher Caud-": "Christopher Caudwell",
    "Van Der Zee": "James Van Der Zee",
    "SUZANNE HARRIS": "Suzanne Harris",   
    "MIKINA KUBINNS": "Mikina Kubinns",
    "BERTA NAVARRO": "Berta Navarro",
    "NORA DE IZQUÉ": "Nora de Izqué",
    "JOSEFINA JORDAN": "Josefina Jordan",
    "BRENDA MARTINEZ": "Brenda Martinez",
    "TIZUKA YAMASAKI": "Tizuka Yamasaki",
    "ANGELINA VANSQUEZ": "Angelina Vansquez",
    "irginia Gray": "Virginia Gray",
    "HHerbert Read": "Herbert Read",
    "H. W. Janson": "Horst Woldemar Janson",
    "Dina Bursztyn Y": "Dina Bursztyn",
    "PENNY GOODFRIEND": "Penny Goodfriend",
    "ELAIN CHRISTENSEN": "Elain Christensen",
    "CLARE COSS": "Clare Coss",
    "BLANCHE WIESEN COOK": "Blanche Wiesen Cook",  
    "SUSAN ORTEGA": "Susan Ortega",
    "June Beer´s": "June Beer",
    "BC.-Kathy Goldman": "Kathy Goldman",
    "Jacki - Richards": "Jacki Richards",
    "The George Sand": "George Sand",
    "de Burcas": "Marin de Burca",
    "ALICE DICK": "Alice Dick", 
    "Ruth Crawfords": "Ruth Crawford Seeger",
    "Ruth Crawford": "Ruth Crawford Seeger",
    "Ruth Craw": "Ruth Crawford Seeger",
    "carolyn johnson": "Carolyn Johnson",
    "Cinda Carr": "Cindy Carr",
    "TISA CHANG": "Tisa Chang",
    "CHERYL CLARKE": "Cheryl Clarke",
    "BREENA CLARKE": "Breena Clarke",
    "LINDA POWELL": "Linda Powell",
    "GWENDOLEN HARDWICK": "Gwendolen Hardwick",
    "LINDA MUSSMANN": "Linda Mussmann",
    "ANN WILSON": "Ann Wilson",
    "Nabokov": "Vladimir Nabokov",
    "Audre Torde´s new": "Audre Lorde",
    "nancy tomlinson ball rice": "Nancy Tomlinson", 
    "Jeanne Piaubert´s": "Jeanne Piaubert",
    "Minh-ha-I": "Trinh Minh-ha", 
}
import re

def clean_name(name):
    # strip leading/trailing punctuation like hyphens, commas, quotes
    name = re.sub(r'^[^a-zA-ZÀ-ÿ]+', '', name)  # remove non-letters from start
    name = re.sub(r'[^a-zA-ZÀ-ÿ]+$', '', name)  # remove non-letters from end
    name = name.strip()
    
    if not name:
        return None
    if name in false_positives:
        return None
    return name_mapping.get(name, name)
for df_ in [df_para, df_refs]:
    df_["persons"] = df_["persons"].apply(
        lambda names: [n for n in (clean_name(n) for n in names) if n is not None]
    )

# combine back into one dataframe
df_combined = pd.concat([df_para, df_refs])

#### 8. One row per person mention

In [68]:
rows = []
for _, row in df_combined.iterrows():
    for person in row["persons"]:
        rows.append({
            "source_file": row["source_file"],
            "issue":       row["issue"],
            "volume":      row["volume"],
            "region_type": row["region_type"],
            "person":      person,
        })

df_persons = pd.DataFrame(rows)
print("Shape:", df_persons.shape)
print(df_persons.head(10))

Shape: (1306, 5)
                                         source_file  issue          volume  \
0  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
1  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
2  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
3  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
4  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
5  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
6  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
7  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
8  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
9  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   

  region_type                person  
0   paragraph    Lorenza C. Schmidt  
1   paragraph       Sylvia Gonzales  

#### 9. Saving

In [69]:
out_path = "/Users/sophiehamann/master-thesis-code/data/processed/09_h2_persons.csv"
df_persons.to_csv(out_path, index=False)
print("Saved to:", out_path)

Saved to: /Users/sophiehamann/master-thesis-code/data/processed/09_h2_persons.csv
